# SVHN Benchmark: MLP, RNN, CNN on Colab & Kaggle


## How to use
1. **Select accelerator** in your notebook settings (GPU for GPU runs). The notebook can still force **CPU** even on a GPU runtime.
2. Edit the **parameters** in the next cell. For the full factorial at one platform, use `fractions = [0.5, 1.0]` and keep both `cpu` and `gpu` in `hardware_list`.
3. Results append to `collab_svhn_benchmark_results.csv`.


In [1]:
# Environment & platform detection
from datetime import datetime, timezone
import os, sys, platform, random, time, subprocess
import numpy as np
import torch

def detect_platform():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ: return 'KAGGLE'
    if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ: return 'COLAB'
    return 'LOCAL'

def get_cpu_name():
    try:
        # Retrieves the specific CPU model name on Linux (Colab/Kaggle)
        return subprocess.check_output("grep -m 1 'model name' /proc/cpuinfo | cut -d: -f2", shell=True).decode().strip()
    except Exception:
        # Fallback for other environments
        return platform.processor() or "Unknown CPU"

PLATFORM = detect_platform()
CPU_NAME = get_cpu_name()
CUDA_AVAILABLE = torch.cuda.is_available()
AUTO_DEVICE = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

print('Platform:', PLATFORM)
print('CPU Model:', CPU_NAME)
print('CUDA available:', CUDA_AVAILABLE, '| Auto device:', AUTO_DEVICE)
if CUDA_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))

Platform: COLAB
CPU Model: Intel(R) Xeon(R) CPU @ 2.00GHz
CUDA available: True | Auto device: cuda
GPU: Tesla T4


In [2]:
# Ensure SciPy (required by torchvision.datasets.SVHN) is available
try:
    import scipy
except Exception:
    print('Installing SciPy...')
    %pip -q install scipy
    import scipy


In [ ]:
# ==== Experiment parameters (edit here) ====
fractions = [0.5, 1.0]   # dataset sizes to benchmark (50% vs 100%)
epochs = 5
batch_size = 64
repeats = 1
num_workers = 2
plots = True
save_metrics = 'collab_svhn_benchmark_results.csv'
seed = 42
models_to_run = ['mlp', 'rnn', 'cnn']
hardware_list = ['cpu', 'gpu']
out_dir = 'outputs'
data_dir = './data'


In [4]:
# Imports & helpers
import os, csv, math
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def set_seeds(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s);
    torch.cuda.manual_seed_all(s);
    torch.backends.cudnn.deterministic = True;
    torch.backends.cudnn.benchmark = False

class MLPClassifier(nn.Module):
    def __init__(self, input_size=3*32*32, hidden_units=256, num_classes=10):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, hidden_units), nn.ReLU(),
            nn.Linear(hidden_units, hidden_units), nn.ReLU(),
            nn.Linear(hidden_units, num_classes)
        )
    def forward(self, x): return self.model(x)

class RNNClassifier(nn.Module):
    def __init__(self, input_size=96, hidden_units=128, num_layers=1, num_classes=10):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_units, num_layers=num_layers, batch_first=True, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_units, num_classes)
    def forward(self, x):
        x = x.permute(0,2,3,1)              # (B,C,H,W)->(B,H,W,C)
        x = x.reshape(x.size(0), 32, 32*3)  # seq_len=32, features=96
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        return self.fc(last)

class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.drop = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64*8*8, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = torch.flatten(x,1); x = self.drop(x)
        x = F.relu(self.fc1(x)); return self.fc2(x)

def get_transforms():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomGrayscale(p=0.1),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
    ])

def make_datasets(root, dataset_fraction, seed):
    tfm = get_transforms()
    full_train = datasets.SVHN(root=root, split='train', transform=tfm, download=True)
    test_ds = datasets.SVHN(root=root, split='test', transform=tfm, download=True)
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(full_train), generator=g).tolist()
    take = int(len(idx) * dataset_fraction)
    sub = Subset(full_train, idx[:take])
    trn_size = int(0.8 * len(sub))
    val_size = len(sub) - trn_size
    train_ds, val_ds = random_split(sub, [trn_size, val_size], generator=g)
    return train_ds, val_ds, test_ds

def get_loaders(train_ds, val_ds, test_ds, batch_size, device_kind, num_workers):
    pin = (device_kind=='gpu')
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    return train_loader, val_loader, test_loader

def train_one_epoch(model, device, loader, optimizer, criterion):
    model.train(); run_loss=0.0; correct=0; total=0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward(); optimizer.step()
        run_loss += loss.item()*x.size(0)
        _,pred = torch.max(out,1)
        correct += (pred==y).sum().item(); total += y.size(0)
    return (run_loss/total if total else 0.0), (100.0*correct/total if total else 0.0)

def evaluate(model, device, loader, criterion):
    model.eval(); run_loss=0.0; correct=0; total=0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            run_loss += loss.item()*x.size(0)
            _,pred = torch.max(out,1)
            correct += (pred==y).sum().item(); total += y.size(0)
    return (run_loss/total if total else 0.0), (100.0*correct/total if total else 0.0)

def plot_losses(train_losses, val_losses, out_path):
    plt.figure(figsize=(10,4)); plt.plot(train_losses,label='Train'); plt.plot(val_losses,label='Val');
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.tight_layout(); plt.savefig(out_path); plt.close()

def plot_confusion(model, device, loader, classes, out_path):
    model.eval(); P=[]; T=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(device); out=model(x); P.extend(out.argmax(1).cpu().numpy().tolist()); T.extend(y.numpy().tolist())
    cm = confusion_matrix(T,P); disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    fig, ax = plt.subplots(figsize=(6,6)); disp.plot(xticks_rotation='vertical', ax=ax, cmap='Blues', colorbar=False)
    plt.tight_layout(); plt.savefig(out_path); plt.close()


In [5]:
# Benchmark loop: runs (hardware × fractions × models × repeats) and logs CSV
from pathlib import Path
Path(out_dir).mkdir(parents=True, exist_ok=True)
classes = [str(i) for i in range(10)]

header = ['timestamp','platform_hint','device_kind','device_name','torch_version','cuda_available','cuda_version',
          'model','dataset_fraction','epochs','batch_size','train_samples','val_samples','test_samples',
          'repeat_idx','seed','total_train_time_s','per_epoch_time_s_mean','throughput_samples_per_s',
          'final_train_loss','final_train_acc','final_val_loss','final_val_acc','test_loss','test_acc']
write_header = not os.path.exists(save_metrics)
with open(save_metrics, 'a', newline='') as f:
    wr = csv.writer(f)
    if write_header: wr.writerow(header)

    for hw in hardware_list:
        device_kind = 'gpu' if hw=='gpu' and torch.cuda.is_available() else ('cpu' if hw=='cpu' else 'skip')
        if hw=='gpu' and not torch.cuda.is_available():
            print('[SKIP] GPU selected but CUDA not available.')
            continue
        device = torch.device('cuda') if device_kind=='gpu' else torch.device('cpu')
        name = torch.cuda.get_device_name(0) if device_kind=='gpu' else CPU_NAME
        print('=== Hardware: {} ({}) ==='.format(device_kind.upper(), name))

        for frac in fractions:
            set_seeds(seed)
            train_ds, val_ds, test_ds = make_datasets(data_dir, frac, seed)
            train_loader, val_loader, test_loader = get_loaders(train_ds, val_ds, test_ds, batch_size, device_kind, num_workers)

            for model_name in models_to_run:
                for rep in range(1, repeats+1):
                    set_seeds(seed + rep - 1)
                    if model_name=='mlp': model = MLPClassifier().to(device)
                    elif model_name=='rnn': model = RNNClassifier().to(device)
                    elif model_name=='cnn': model = CNNClassifier().to(device)
                    else: raise ValueError(model_name)

                    criterion = nn.CrossEntropyLoss()
                    optim = torch.optim.Adam(model.parameters(), lr=1e-3)
                    epoch_times = []
                    tr_losses, va_losses = [], []

                    t0 = time.perf_counter()
                    for ep in range(1, epochs+1):
                        e0 = time.perf_counter()
                        tr_loss, tr_acc = train_one_epoch(model, device, train_loader, optim, criterion)
                        if device_kind=='gpu': torch.cuda.synchronize()
                        e1 = time.perf_counter(); epoch_times.append(e1-e0)
                        va_loss, va_acc = evaluate(model, device, val_loader, criterion)
                        tr_losses.append(tr_loss); va_losses.append(va_loss)
                        print('[{}][Rep {}] Epoch {}/{} | Train {:.4f}/{:.2f}% | Val {:.4f}/{:.2f}% | Time {:.2f}s'.format(
                              model_name.upper(), rep, ep, epochs, tr_loss, tr_acc, va_loss, va_acc, epoch_times[-1]))

                    if device_kind=='gpu': torch.cuda.synchronize()
                    total_train_time = time.perf_counter() - t0
                    test_loss, test_acc = evaluate(model, device, test_loader, criterion)
                    per_epoch_mean = float(np.mean(epoch_times)) if epoch_times else 0.0
                    throughput = (len(train_loader.dataset)*epochs)/total_train_time if total_train_time>0 else 0.0

                    # Optional plots
                    if plots:
                        base = os.path.join(out_dir, '{}_{}_rep{}_frac{}'.format(model_name, device_kind, rep, frac))
                        plot_losses(tr_losses, va_losses, base + '_loss.png')
                        plot_confusion(model, device, val_loader, classes, base + '_confusion_val.png')

                    wr.writerow([
                        datetime.now(timezone.utc).isoformat(),
                        os.environ.get('KAGGLE_KERNEL_RUN_TYPE','COLAB' if 'COLAB_GPU' in os.environ else 'LOCAL'),
                        device_kind, name, torch.__version__, torch.cuda.is_available(),
                        (torch.version.cuda if torch.version.cuda is not None else ''),
                        model_name, frac, epochs, batch_size, len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset),
                        rep, seed + rep - 1, round(total_train_time,4), round(per_epoch_mean,4), round(throughput,2),
                        round(tr_losses[-1],4), round(tr_acc,2), round(va_losses[-1],4), round(va_acc,2), round(test_loss,4), round(test_acc,2)
                    ])
                    f.flush()
                    print('[DONE] {} | HW {} | Rep {} | Total {:.2f}s | Throughput {:.2f} samples/s | ValAcc {:.2f}% | TestAcc {:.2f}%'.format(
                          model_name.upper(), device_kind, rep, total_train_time, throughput, va_acc, test_acc))

print('Results appended to:', save_metrics)


=== Hardware: CPU (Intel(R) Xeon(R) CPU @ 2.00GHz) ===


100%|██████████| 182M/182M [00:01<00:00, 94.8MB/s]
100%|██████████| 64.3M/64.3M [00:01<00:00, 49.9MB/s]


[MLP][Rep 1] Epoch 1/5 | Train 1.4478/51.80% | Val 1.0376/67.79% | Time 12.09s
[MLP][Rep 1] Epoch 2/5 | Train 0.9795/69.33% | Val 0.9146/71.18% | Time 11.82s
[MLP][Rep 1] Epoch 3/5 | Train 0.8429/73.77% | Val 0.8799/72.37% | Time 11.91s
[MLP][Rep 1] Epoch 4/5 | Train 0.7571/76.08% | Val 0.7840/75.48% | Time 14.26s
[MLP][Rep 1] Epoch 5/5 | Train 0.6853/78.63% | Val 0.7723/76.43% | Time 12.06s
[DONE] MLP | HW cpu | Rep 1 | Total 73.27s | Throughput 1999.51 samples/s | ValAcc 76.43% | TestAcc 73.43%
[RNN][Rep 1] Epoch 1/5 | Train 2.2113/20.29% | Val 2.2601/18.02% | Time 13.05s
[RNN][Rep 1] Epoch 2/5 | Train 2.1120/23.79% | Val 1.9513/30.63% | Time 13.38s
[RNN][Rep 1] Epoch 3/5 | Train 1.8544/34.53% | Val 1.8164/37.50% | Time 13.14s
[RNN][Rep 1] Epoch 4/5 | Train 1.6903/40.76% | Val 1.6517/41.70% | Time 13.31s
[RNN][Rep 1] Epoch 5/5 | Train 1.5485/45.85% | Val 1.4802/49.29% | Time 13.21s
[DONE] RNN | HW cpu | Rep 1 | Total 78.60s | Throughput 1863.89 samples/s | ValAcc 49.29% | TestAcc 47.

In [6]:
# Summarize results: mean±std for time and accuracy, and GPU speedup
import pandas as pd
df = pd.read_csv(save_metrics)
display(df.tail(10))

# Grouped summary
grp = df.groupby(['platform_hint','device_kind','model','dataset_fraction']).agg(
    runs=('repeat_idx','count'),
    total_time_mean=('total_train_time_s','mean'), total_time_std=('total_train_time_s','std'),
    per_epoch_mean=('per_epoch_time_s_mean','mean'),
    throughput_mean=('throughput_samples_per_s','mean'),
    val_acc_mean=('final_val_acc','mean'), val_acc_std=('final_val_acc','std'),
    test_acc_mean=('test_acc','mean'), test_acc_std=('test_acc','std')
).reset_index()
display(grp.sort_values(['model','dataset_fraction','device_kind']))

# Compute GPU speedup vs CPU by matching rows
pivot = grp.pivot_table(index=['platform_hint','model','dataset_fraction'], columns='device_kind', values='total_time_mean')
if 'cpu' in pivot.columns and 'gpu' in pivot.columns:
    pivot['gpu_speedup'] = pivot['cpu'] / pivot['gpu']
    display(pivot)
else:
    print('GPU or CPU results missing; run both hardware configs.')


,timestamp,platform_hint,device_kind,device_name,torch_version,cuda_available,cuda_version,model,dataset_fraction,epochs,...,seed,total_train_time_s,per_epoch_time_s_mean,throughput_samples_per_s,final_train_loss,final_train_acc,final_val_loss,final_val_acc,test_loss,test_acc
2,2026-03-09T00:18:36.946711+00:00,COLAB,cpu,Intel(R) Xeon(R) CPU @ 2.00GHz,2.10.0+cu128,True,12.8,cnn,0.5,5,...,42,239.0054,42.0507,613.00,0.3927,88.24,0.3984,88.67,0.4782,86.30
3,2026-03-09T00:21:11.761858+00:00,COLAB,cpu,Intel(R) Xeon(R) CPU @ 2.00GHz,2.10.0+cu128,True,12.8,mlp,1.0,5,...,42,138.0017,22.9565,2123.34,0.5932,81.89,0.6660,80.49,0.8110,77.66
4,2026-03-09T00:24:02.255027+00:00,COLAB,cpu,Intel(R) Xeon(R) CPU @ 2.00GHz,2.10.0+cu128,True,12.8,rnn,1.0,5,...,42,156.2821,26.4138,1874.97,1.2355,57.87,1.2186,58.54,1.2569,57.53
5,2026-03-09T00:32:00.982848+00:00,COLAB,cpu,Intel(R) Xeon(R) CPU @ 2.00GHz,2.10.0+cu128,True,12.8,cnn,1.0,5,...,42,448.0483,78.8303,654.00,0.3546,89.34,0.3550,89.80,0.4039,88.27
6,2026-03-09T00:33:10.668611+00:00,COLAB,gpu,Tesla T4,2.10.0+cu128,True,12.8,mlp,0.5,5,...,42,55.9350,8.9071,2619.29,0.6849,78.61,0.7561,76.95,0.9724,72.96
7,2026-03-09T00:34:17.157352+00:00,COLAB,gpu,Tesla T4,2.10.0+cu128,True,12.8,rnn,0.5,5,...,42,56.1559,8.9697,2608.99,1.5219,47.27,1.4849,48.65,1.5368,47.73
8,2026-03-09T00:35:29.891313+00:00,COLAB,gpu,Tesla T4,2.10.0+cu128,True,12.8,cnn,0.5,5,...,42,61.8914,10.2260,2367.21,0.3974,88.05,0.4274,87.59,0.4967,85.29
9,2026-03-09T00:37:39.953519+00:00,COLAB,gpu,Tesla T4,2.10.0+cu128,True,12.8,mlp,1.0,5,...,42,114.1099,18.3671,2567.92,0.5948,81.72,0.6729,80.33,0.8352,77.04
10,2026-03-09T00:39:49.871553+00:00,COLAB,gpu,Tesla T4,2.10.0+cu128,True,12.8,rnn,1.0,5,...,42,116.8124,18.7351,2508.51,1.1967,60.29,1.3479,55.66,1.4532,52.09
11,2026-03-09T00:42:01.639158+00:00,COLAB,gpu,Tesla T4,2.10.0+cu128,True,12.8,cnn,1.0,5,...,42,119.1927,19.4768,2458.41,0.3485,89.51,0.3510,90.00,0.4066,88.10


,platform_hint,device_kind,model,dataset_fraction,runs,total_time_mean,total_time_std,per_epoch_mean,throughput_mean,val_acc_mean,val_acc_std,test_acc_mean,test_acc_std
0,COLAB,cpu,cnn,0.5,1,239.0054,NaN,42.0507,613.00,88.67,NaN,86.30,NaN
6,COLAB,gpu,cnn,0.5,1,61.8914,NaN,10.2260,2367.21,87.59,NaN,85.29,NaN
1,COLAB,cpu,cnn,1.0,1,448.0483,NaN,78.8303,654.00,89.80,NaN,88.27,NaN
7,COLAB,gpu,cnn,1.0,1,119.1927,NaN,19.4768,2458.41,90.00,NaN,88.10,NaN
2,COLAB,cpu,mlp,0.5,1,73.2729,NaN,12.4261,1999.51,76.43,NaN,73.43,NaN
8,COLAB,gpu,mlp,0.5,1,55.9350,NaN,8.9071,2619.29,76.95,NaN,72.96,NaN
3,COLAB,cpu,mlp,1.0,1,138.0017,NaN,22.9565,2123.34,80.49,NaN,77.66,NaN
9,COLAB,gpu,mlp,1.0,1,114.1099,NaN,18.3671,2567.92,80.33,NaN,77.04,NaN
4,COLAB,cpu,rnn,0.5,1,78.6043,NaN,13.2172,1863.89,49.29,NaN,47.31,NaN
10,COLAB,gpu,rnn,0.5,1,56.1559,NaN,8.9697,2608.99,48.65,NaN,47.73,NaN


device_kind                                cpu       gpu  gpu_speedup
platform_hint model dataset_fraction                                 
COLAB         cnn   0.5               239.0054   61.8914     3.861690
                    1.0               448.0483  119.1927     3.759025
              mlp   0.5                73.2729   55.9350     1.309965
                    1.0               138.0017  114.1099     1.209375
              rnn   0.5                78.6043   56.1559     1.399751
                    1.0               156.2821  116.8124     1.337890

---
### Notes
- TorchVision's SVHN loader remaps the digit '0' to label **0** (the original dataset uses **10** for '0'), keeping class labels in **[0..9]** for `CrossEntropyLoss`.
- Loading SVHN `.mat` files via TorchVision requires **SciPy**.
- You can benchmark both **CPU and GPU** in a single GPU-enabled runtime; CPU runs are forced by selecting `device='cpu'`.